In [1]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import tensorflow as tf
imagegen = tf.keras.preprocessing.image.ImageDataGenerator
mobilenet = tf.keras.applications.MobileNetV2
dense = tf.keras.layers.Dense
globalpool = tf.keras.layers.GlobalAveragePooling2D
model = tf.keras.models.Model

In [2]:
imgsize = 224
batchsize = 32

gen = imagegen(
    rescale=1.0/255,
    validation_split=0.2,
    rotation_range=20,
    horizontal_flip=True,
    zoom_range=0.15,
    brightness_range=[0.8, 1.2]
)

traingen = gen.flow_from_directory(
    'dataset',
    target_size=(imgsize, imgsize),
    batch_size=batchsize,
    class_mode='binary',
    subset='training'
)

valgen = gen.flow_from_directory(
    'dataset',
    target_size=(imgsize, imgsize),
    batch_size=batchsize,
    class_mode='binary',
    subset='validation'
)

print(traingen.class_indices)

Found 74 images belonging to 2 classes.
Found 18 images belonging to 2 classes.
{'healthy_tomato': 0, 'tomato_mosaic_virus': 1}


In [3]:
base = mobilenet(input_shape=(imgsize, imgsize, 3), include_top=False, weights='imagenet')
base.trainable = False

top = base.output
top = globalpool()(top)
out = dense(1, activation='sigmoid')(top)

net = model(inputs=base.input, outputs=out)
net.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 56, 56,    │      2,304 │ block_1_depthwis

 Total params: 2,259,265 (8.62 MB)

 Trainable params: 1,281 (5.00 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [4]:
net.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [5]:
stopearly = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

hist = net.fit(
    traingen,
    validation_data=valgen,
    epochs=20,
    callbacks=[stopearly]
)

Epoch 1/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 29s 5s/step - accuracy: 0.3378 - loss: 1.3041 - val_accuracy: 0.2778 - val_loss: 0.9014
Epoch 2/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 7s 2s/step - accuracy: 0.3649 - loss: 0.8726 - val_accuracy: 0.5556 - val_loss: 0.8237
Epoch 3/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 7s 3s/step - accuracy: 0.6757 - loss: 0.7353 - val_accuracy: 0.6667 - val_loss: 0.7225
Epoch 4/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 6s 2s/step - accuracy: 0.6486 - loss: 0.7525 - val_accuracy: 0.6667 - val_loss: 0.6558
Epoch 5/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 10s 2s/step - accuracy: 0.6622 - loss: 0.6510 - val_accuracy: 0.6667 - val_loss: 0.5999
Epoch 6/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 7s 2s/step - accuracy: 0.6622 - loss: 0.6200 - val_accuracy: 0.7222 - val_loss: 0.5775
Epoch 7/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 6s 3s/step - accuracy: 0.7703 - loss: 0.4880 - val_accuracy: 0.9444 - val_loss: 0.4683
Epoch 8/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 6s 2s/step - accuracy: 0.8784 - loss: 0.4579 - val_accuracy: 0.9444 - val_loss: 0.4562
Epoch 9/20
3/3

In [11]:
net.save('tomato_model.keras')

In [6]:
from PIL import Image
import numpy as np

def loadall(folder):
    imgs = []
    for cls in os.listdir(folder):
        clspath = os.path.join(folder, cls)
        for fname in os.listdir(clspath):
            path = os.path.join(clspath, fname)
            img = Image.open(path).convert("RGB").resize((imgsize, imgsize))
            imgs.append(np.array(img, dtype="float32") / 255.0)
    return np.array(imgs)

alldata = loadall("dataset")
print(alldata.shape)

(92, 224, 224, 3)


In [7]:
from sklearn.model_selection import train_test_split

aetrain, aeval = train_test_split(alldata, test_size=0.2, random_state=42)
print(aetrain.shape, aeval.shape)

(73, 224, 224, 3) (19, 224, 224, 3)


In [8]:
encoderinput = tf.keras.layers.Input(shape=(imgsize, imgsize, 3))

x = tf.keras.layers.Conv2D(16, (3, 3), activation="relu", padding="same")(encoderinput)
x = tf.keras.layers.MaxPooling2D(padding="same")(x)
x = tf.keras.layers.Conv2D(8, (3, 3), activation="relu", padding="same")(x)
encoded = tf.keras.layers.MaxPooling2D(padding="same")(x)

x = tf.keras.layers.Conv2D(8, (3, 3), activation="relu", padding="same")(encoded)
x = tf.keras.layers.UpSampling2D()(x)
x = tf.keras.layers.Conv2D(16, (3, 3), activation="relu", padding="same")(x)
x = tf.keras.layers.UpSampling2D()(x)
decoded = tf.keras.layers.Conv2D(3, (3, 3), activation="sigmoid", padding="same")(x)

autoencoder = tf.keras.models.Model(encoderinput, decoded)
autoencoder.compile(optimizer="adam", loss="mse")
autoencoder.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 224, 224, 16)   │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 112, 112, 16)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 112, 112, 8)    │         1,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 56, 56, 8)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 56, 56, 8)      │           584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ up_sampling2d (UpSampling2D)    │ (None, 112, 112, 8)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 112, 112, 16)   │         1,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ up_sampling2d_1 (UpSampling2D)  │ (None, 224, 224, 16)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 224, 224, 3)    │           435 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,795 (14.82 KB)

 Trainable params: 3,795 (14.82 KB)

 Non-trainable params: 0 (0.00 B)

In [9]:
autoencoder.fit(
    aetrain, aetrain,
    validation_data=(aeval, aeval),
    epochs=30,
    batch_size=8,
    verbose=1
)

Epoch 1/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 12s 504ms/step - loss: 0.0250 - val_loss: 0.0256
Epoch 2/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 3s 304ms/step - loss: 0.0211 - val_loss: 0.0204
Epoch 3/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 3s 270ms/step - loss: 0.0161 - val_loss: 0.0151
Epoch 4/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 3s 283ms/step - loss: 0.0140 - val_loss: 0.0127
Epoch 5/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 3s 296ms/step - loss: 0.0131 - val_loss: 0.0121
Epoch 6/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 3s 268ms/step - loss: 0.0114 - val_loss: 0.0114
Epoch 7/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 3s 292ms/step - loss: 0.0104 - val_loss: 0.0104
Epoch 8/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 3s 258ms/step - loss: 0.0091 - val_loss: 0.0104
Epoch 9/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 3s 289ms/step - loss: 0.0086 - val_loss: 0.0098
Epoch 10/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 3s 261ms/step - loss: 0.0080 - val_loss: 0.0083
Epoch 11/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 3s 288ms/step - loss: 0.0074 - val_loss: 0.0080
Epoch 12/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 3s 260ms/ste

In [10]:
reconval = autoencoder.predict(aeval)
errorsval = np.mean(np.square(aeval - reconval), axis=(1, 2, 3))

threshold = np.mean(errorsval) + 2 * np.std(errorsval)
print(f"mean error {np.mean(errorsval):.5f}, std {np.std(errorsval):.5f}")
print(f"anomaly threshold {threshold:.5f}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
mean error 0.00603, std 0.00277
anomaly threshold 0.01156


In [11]:
autoencoder.save("tomatoAnomalyDetector.keras")

with open("tomatoAnomalyThreshold.txt", "w") as f:
    f.write(str(threshold))